
<a href="https://colab.research.google.com/github/go-fair-us/apireference/blob/master/code/nde_viz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



In [1]:
import pandas as pd
import networkx as nx
import pyoxigraph
# import requests
import threading
from ipysigma import Sigma
from pyld import jsonld
from pyoxigraph import RdfFormat
import json
import sys
import nest_asyncio
from playwright.sync_api import sync_playwright
nest_asyncio.apply()  # allows nested event loops in Colab

In [2]:
def get_json_ld(url: str) -> str:
    results = []

    def run():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            page.goto(url, wait_until="networkidle")

            json_ld_docs = page.evaluate("""() => {
                const scripts = document.querySelectorAll('script[type="application/ld+json"]');
                return Array.from(scripts).map(s => {
                    try { return JSON.parse(s.textContent); }
                    catch { return null; }
                }).filter(Boolean);
            }""")

            browser.close()
            results.extend(json_ld_docs)

    thread = threading.Thread(target=run)
    thread.start()
    thread.join()
    return results[0]


In [3]:
def load_jsonld_to_oxigraph(store, df, column: str):
    """
    Load JSON-LD strings from a DataFrame column into an Oxigraph store.
    Expands remote @context references via pyld before loading.
    """

    for _, row in df.iterrows():
        raw = row[column]
        if not raw:
            continue

        if isinstance(raw, dict):
            doc = raw  # already a dict, no need to serialize/deserialize
        else:
            doc = json.loads(raw)

        doc['@context'] = "https://schema.org/docs/jsonldcontext.json"


        # Expand resolves all remote @context URLs, then re-serialize to N-Quads
        # expanded = jsonld.expand(doc)
        nquads = jsonld.to_rdf(doc, {"format": "application/n-quads"})

        store.load(nquads, RdfFormat.TURTLE, base_iri=None, to_graph=None)



In [4]:

with open("./input/NDE_Resource_list.txt", "r") as f:
    nde_resources = pd.DataFrame(
        [line.strip() for line in f if line.strip()],
        columns=["url"]
    )

print(f"Loaded {len(nde_resources)} resources")
# nde_resources


Loaded 4 resources


In [5]:
url = nde_resources.iloc[0]["url"]
results = get_json_ld(url)
print(json.dumps(results, indent=2))


{
  "@type": "ResourceCatalog",
  "about": [
    {
      "@type": "DefinedTerm",
      "description": "For the schema, consider https://bioschemas.org/types/Phenotype",
      "displayName": "Phenotype",
      "name": "Phenotype",
      "url": "http://purl.obolibrary.org/obo/NCIT_C16977"
    },
    {
      "@type": "DefinedTerm",
      "description": "Not distinct from Specimen as a schema. For schema consider, https://bioschemas.org/types/BioSample/",
      "displayName": "Sample",
      "name": "Sample",
      "url": "http://www.w3.org/ns/sosa/Sample"
    },
    {
      "@type": "DefinedTerm",
      "description": "For schema, consider https://schema.org/Taxon",
      "displayName": "Taxon",
      "name": "Taxon",
      "url": "http://purl.obolibrary.org/obo/NCIT_C40098"
    }
  ],
  "abstract": "BacDive is a BMBF supported IID repository that includes phenotypic data.",
  "alternateName": "Bacterial Diversity Metadatabase",
  "author": [
    {
      "@type": "Person",
      "affiliat

In [6]:

nde_resources["jsonld"] = nde_resources["url"].apply(get_json_ld)
# nde_resources[["url", "jsonld"]]

In [7]:
mem_store = pyoxigraph.Store()

load_jsonld_to_oxigraph(mem_store, nde_resources, "jsonld")

In [8]:
# SPARQL
rq = """
PREFIX schema: <http://schema.org/>

SELECT  ?s ?name ?kw ?license ?tgurl
 WHERE {
     ?s a schema:ResourceCatalog .
     ?s schema:name ?name .
     ?s schema:keywords ?kw .
     ?s schema:license ?license .
     ?s schema:topicCategory ?tg .
     ?tg schema:url ?tgurl .

 }
"""


In [9]:
# This map is necessary to get the values from the pyoxigraph.QuerySolutions
def extract_value(cell):
    if isinstance(cell, (pyoxigraph.Literal, pyoxigraph.NamedNode, pyoxigraph.BlankNode)):
        return cell.value
    return cell

r = mem_store.query(rq)
results = list(r)

vars = r.variables
value_list = [variable.value for variable in vars]

rq_df = pd.DataFrame(results, columns=value_list)
rq_df = rq_df.map(extract_value)


In [10]:
rq_df


,s,name,kw,license,tgurl
0,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Preclinical and clinical studies,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0625
1,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Preclinical and clinical studies,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0622
2,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Preclinical and clinical studies,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0199
3,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Pharmacogenomics,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0625
4,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Pharmacogenomics,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0622
5,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Pharmacogenomics,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0199
6,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Patient Care,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0625
7,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Patient Care,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0622
8,b01b877732e99862d4a52074446af199,Clinical Genome Resource,Patient Care,https://creativecommons.org/publicdomain/zero/...,http://edamontology.org/topic_0199
9,b5d0ca81ead0188920a045955efe3847,ClinVar,Pharmacogenomics,https://www.ncbi.nlm.nih.gov/clinvar/intro/,http://edamontology.org/topic_0625


In [11]:
G = nx.DiGraph()

for _, row in rq_df.iterrows():
    creator = str(row["name"]).strip() if pd.notna(row["name"]) else None
    kw = str(row["kw"]).strip() if pd.notna(row["kw"]) else None
    # funder = str(row["tgurl"]).strip() if pd.notna(row["tgurl"]) else None

    if not creator:
        continue

    # Add nodes with type (so we can color them differently)
    G.add_node(creator, node_type="creator", label=creator)

    if kw:
        G.add_node(kw, node_type="keyword", label=kw)
        G.add_edge(creator, kw, relation="connects_to")
    #
    # if funder:
    #     G.add_node(funder, node_type="funder", label=funder)
    #     G.add_edge(creator, funder, relation="funded_by")

# Optional: add degree for sizing
degrees = dict(G.degree())
for node in G.nodes():
    G.nodes[node]["degree"] = degrees[node] + 3  # +3 for minimum visibility

print(f"Built graph with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges")

Built graph with 11 nodes and 9 edges


In [12]:
# For Google Colab
if 'google.colab' in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()
else:
    print("Not in Colab - skipping installs.")

Not in Colab - skipping installs.


In [13]:
Sigma(
    G,
    node_color="node_type",           # colors creators blue, keywords orange, funders green (auto palette)
    node_size="degree",               # bigger = more connections
    edge_color="relation",            # different colors for "connects_to" vs "funded_by"
    node_label="label",
    height=800,
    start_layout=True                 # runs force-directed layout automatically
)

Sigma(nx.DiGraph with 11 nodes and 9 edges)